## 1. Define Ingestion Function
Hàm generic để ingest dữ liệu từ CSV sang Delta Table.
- **Input:** Tên file CSV, tên bảng đích.
- **Logic:** Read Stream (Auto Loader) -> Add Metadata -> Write Stream (Delta).

In [0]:
from pyspark.sql.functions import current_timestamp, col

def ingest_bronze_table(source_filename, table_name, schema_hints=None):

    source_dir = raw_data_path 
    
    checkpoint_loc = f"{checkpoint_base_path}/{table_name}"
    schema_loc = f"{checkpoint_base_path}/{table_name}/schema"
    
    print(f"Processing: {source_filename} -> Table: {table_name}")
    
    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", schema_loc)
        .option("pathGlobFilter", source_filename)
        .option("header", "true")
        .option("delimiter", ",")
        .option("quote", "\"")
        .option("multiline", "true")
    )
    
    if schema_hints:
        reader = reader.option("cloudFiles.schemaHints", schema_hints)
        
    df_raw = reader.load(source_dir)
    
    # ✅ FIX UC HERE
    df_enhanced = (
        df_raw
        .withColumn("ingestion_date", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
    )
        
    query = (
        df_enhanced.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_loc)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table(f"{catalog_name}.{schema_name}.{table_name}")
    )
    
    query.awaitTermination()
    print(f"✅ Completed: {table_name}")


## 2. Execute Ingestion
Định nghĩa danh sách file nguồn và tên bảng đích, sau đó chạy vòng lặp để load toàn bộ dữ liệu.

In [0]:
# Dictionary map: "Tên file CSV gốc" -> "Tên bảng đích mong muốn"
tables_to_ingest = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "product_category_name_translation"
}

# Chạy vòng lặp xử lý từng bảng
for filename, tablename in tables_to_ingest.items():
    try:
        # Đối với bảng Reviews, comment thường hay lỗi xuống dòng, cần lưu ý
        ingest_bronze_table(filename, tablename)
    except Exception as e:
        print(f"❌ Error loading {tablename}: {str(e)}")

# 3. Validate Data
Kiểm tra xem dữ liệu đã vào bảng Bronze thành công chưa và xem cấu trúc dữ liệu.

In [0]:
# Liệt kê các bảng đã tạo
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

# Xem thử dữ liệu bảng orders (Bảng quan trọng nhất)
# Bạn sẽ thấy cột 'ingestion_date' và 'source_file' ở cuối
display(spark.read.table(f"{catalog_name}.{schema_name}.orders").limit(10))